# 04B — Positive Spatial-Accuracy Filter and Background Candidate Pool

**FYP:** Developing a Semi-Dynamic Landslide Susceptibility Decision-Support Dashboard Using Machine Learning for Malaysia

## Why this stage is needed

Stage 4 revealed that some NASA COOLR records have coarse positional accuracy:

- 10 km
- 25 km
- 50 km
- 250 km

Fine-resolution predictors such as 30 m slope, land cover, and distance to roads/waterways should not be extracted at a point whose true event location may be tens or hundreds of kilometres away.

Therefore, this notebook creates two separate inventories:

1. **Historical positive inventory** — keeps all reviewed landslide events.
2. **ML-eligible positive inventory** — contains only records spatially precise enough for point-based feature extraction.

## Baseline positional-accuracy rule

For the first model:

- JMG resolved source geometry → INCLUDE
- COOLR exact / ≤5 km location accuracy → INCLUDE
- COOLR >5 km → retain historically but EXCLUDE from first point-based ML training

This is a modelling-quality decision, not deletion of historical data.

## Background candidates

Instead of immediately fixing the final pseudo-absence count, this notebook generates a larger pool of **3,000 spatially valid background candidates**.

The next stage will extract environmental features and remove only invalid candidates such as:
- open water,
- missing DEM,
- invalid/no-data predictor coverage.

Final pseudo-absence selection comes after feature validity checks.


In [2]:
# OPTIONAL: uncomment if needed
# !pip install -q geopandas pandas numpy matplotlib shapely pyproj

from pathlib import Path
import re
import math
import warnings

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt

from shapely.geometry import Point, box
from shapely.ops import unary_union

warnings.filterwarnings("ignore", category=UserWarning)


## 1. Project paths

In [3]:
# ============================================================
# CHANGE ONLY THIS LINE
# ============================================================
PROJECT_ROOT = Path(r"C:\Landslide_FYP")

RAW_DIR = PROJECT_ROOT / "Landslide_FYP_Data"
DEV_DIR = PROJECT_ROOT / "Landslide_FYP_Development"

POSITIVE_PATH = (
    DEV_DIR
    / "02_cleaned_inventory"
    / "malaysia_positive_inventory_final.geojson"
)

BOUNDARY_PATH = (
    RAW_DIR
    / "02_Boundary"
    / "mys_admin0.geojson"
)

OUTPUT_DIR = (
    DEV_DIR
    / "01_preprocessing"
    / "positive_accuracy_background_outputs"
)

CLEAN_DIR = DEV_DIR / "02_cleaned_inventory"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

for name, path in {
    "Final positive inventory": POSITIVE_PATH,
    "Malaysia boundary": BOUNDARY_PATH,
}.items():
    print(name, ":", path)
    if not path.exists():
        raise FileNotFoundError(path)


Final positive inventory : C:\Landslide_FYP\Landslide_FYP_Development\02_cleaned_inventory\malaysia_positive_inventory_final.geojson
Malaysia boundary : C:\Landslide_FYP\Landslide_FYP_Data\02_Boundary\mys_admin0.geojson


## 2. Load data

In [4]:
positive = gpd.read_file(POSITIVE_PATH).to_crs("EPSG:4326")
malaysia = gpd.read_file(BOUNDARY_PATH).to_crs("EPSG:4326")

print("Historical positives:", len(positive))
print(positive["source_dataset"].value_counts())

if len(positive) != 287:
    print("WARNING: previous Stage 3 result was 287 positives.")


Historical positives: 287
source_dataset
NASA_COOLR    235
JMG            52
Name: count, dtype: int64


## 3. Convert COOLR location-accuracy labels to metres

No missing value is guessed as a better accuracy.

For JMG, original government source geometry is treated as precise enough for this prototype.


In [5]:
def accuracy_to_m(value):
    if pd.isna(value):
        return np.nan

    text = str(value).strip().lower()

    if text in {"exact", "precise", "point"}:
        return 0.0

    match = re.search(r"(\d+(?:\.\d+)?)", text)
    if not match:
        return np.nan

    number = float(match.group(1))

    if "km" in text:
        return number * 1000.0
    elif "m" in text:
        return number

    return np.nan


positive["location_accuracy_m"] = np.nan

coolr_mask = positive["source_dataset"].eq("NASA_COOLR")
jmg_mask = positive["source_dataset"].eq("JMG")

positive.loc[coolr_mask, "location_accuracy_m"] = (
    positive.loc[coolr_mask, "location_accuracy_source"]
    .apply(accuracy_to_m)
)

positive.loc[jmg_mask, "location_accuracy_m"] = 0.0

display(
    positive.groupby(
        ["source_dataset", "location_accuracy_source"],
        dropna=False
    )
    .size()
    .reset_index(name="records")
)


,source_dataset,location_accuracy_source,records
0,JMG,<NA>,52
1,NASA_COOLR,10km,31
2,NASA_COOLR,1km,61
3,NASA_COOLR,250km,1
4,NASA_COOLR,25km,34
5,NASA_COOLR,50km,15
6,NASA_COOLR,5km,66
7,NASA_COOLR,exact,21
8,NASA_COOLR,unknown,6


## 4. Apply the baseline ML-eligibility rule

Maximum accepted COOLR positional uncertainty:

**5 km**


In [6]:
MAX_COOLR_ACCURACY_M = 5_000

positive["ml_eligible"] = False
positive["ml_exclusion_reason"] = pd.NA

# JMG source points
positive.loc[jmg_mask, "ml_eligible"] = True

# COOLR with known accuracy <= 5 km
coolr_eligible = (
    coolr_mask
    & positive["location_accuracy_m"].notna()
    & (positive["location_accuracy_m"] <= MAX_COOLR_ACCURACY_M)
)

positive.loc[coolr_eligible, "ml_eligible"] = True

# Coarse COOLR
coarse = (
    coolr_mask
    & positive["location_accuracy_m"].notna()
    & (positive["location_accuracy_m"] > MAX_COOLR_ACCURACY_M)
)

positive.loc[
    coarse,
    "ml_exclusion_reason"
] = "COOLR positional uncertainty > 5 km"

# Unknown COOLR accuracy
unknown = coolr_mask & positive["location_accuracy_m"].isna()

positive.loc[
    unknown,
    "ml_exclusion_reason"
] = "COOLR location accuracy unavailable/uninterpretable"

summary = (
    positive.groupby(
        ["source_dataset", "ml_eligible"],
        dropna=False
    )
    .size()
    .reset_index(name="records")
)

display(summary)

print("Historical positives:", len(positive))
print("ML-eligible positives:", int(positive["ml_eligible"].sum()))
print("Excluded from point-based ML:", int((~positive["ml_eligible"]).sum()))


,source_dataset,ml_eligible,records
0,JMG,True,52
1,NASA_COOLR,False,87
2,NASA_COOLR,True,148


Historical positives: 287
ML-eligible positives: 200
Excluded from point-based ML: 87


## 5. Inspect excluded records

These records are **not deleted**. They remain part of the historical inventory.


In [7]:
excluded = positive.loc[
    ~positive["ml_eligible"],
    [
        "final_positive_id",
        "source_dataset",
        "source_record_id",
        "event_date",
        "state_or_admin",
        "district_or_locality",
        "event_title_or_name",
        "location_accuracy_source",
        "location_accuracy_m",
        "longitude",
        "latitude",
        "ml_exclusion_reason",
    ]
].copy()

display(
    excluded.sort_values(
        "location_accuracy_m",
        ascending=False
    )
)

excluded.to_csv(
    OUTPUT_DIR / "positive_excluded_from_point_ml.csv",
    index=False
)


,final_positive_id,source_dataset,source_record_id,event_date,state_or_admin,district_or_locality,event_title_or_name,location_accuracy_source,location_accuracy_m,longitude,latitude,ml_exclusion_reason
185,POS_0186,NASA_COOLR,COOLR_9317,2016-02-18 00:00:00,Sarawak,Southern Sarawak,Southern Sarawak,250km,250000.0,114.3859,3.3334,COOLR positional uncertainty > 5 km
11,POS_0012,NASA_COOLR,COOLR_5781,2014-01-14 09:00:00,Sabah,Sabah,Sabah,50km,50000.0,116.7968,5.4204,COOLR positional uncertainty > 5 km
112,POS_0113,NASA_COOLR,COOLR_4885,2013-05-29 00:00:00,Sabah,"Jalan Merakau, Ranau","Jalan Merakau, Ranau",50km,50000.0,116.6928,5.9872,COOLR positional uncertainty > 5 km
169,POS_0170,NASA_COOLR,COOLR_4209,2012-02-18 01:30:00,Sabah,"Kampung Terusan, Lahad Datu","Kampung Terusan, Lahad Datu",50km,50000.0,118.4047,5.0882,COOLR positional uncertainty > 5 km
163,POS_0164,NASA_COOLR,COOLR_7609,2015-08-09 06:00:00,Sabah,Kota Kinabalu-Tambunan Road,Kota Kinabalu-Tambunan Road,50km,50000.0,116.2113,5.9012,COOLR positional uncertainty > 5 km
...,...,...,...,...,...,...,...,...,...,...,...,...
104,POS_0105,NASA_COOLR,COOLR_5736,2013-12-30 00:00:00,Sarawak,"Mukah, Sarawak","Mukah, Sarawak",unknown,NaN,112.0786,2.8961,COOLR location accuracy unavailable/uninterpre...
124,POS_0125,NASA_COOLR,COOLR_922,2008-12-11 00:00:00,Terengganu,"Bukit Apil along Jalan Cerung Lanjut, Terengganu","Bukit Apil along Jalan Cerung Lanjut, Terengganu",unknown,NaN,102.9896,5.0936,COOLR location accuracy unavailable/uninterpre...
147,POS_0148,NASA_COOLR,COOLR_6572,2014-12-19 17:00:00,Terengganu,Tasik Kenyir road,Tasik Kenyir road,unknown,NaN,102.6857,5.1894,COOLR location accuracy unavailable/uninterpre...
188,POS_0189,NASA_COOLR,COOLR_6621,2015-01-03 05:00:00,Sabah,"Kampung Timor, in front of the old Police Station","Kampung Timor, in front of the old Police Station",unknown,NaN,118.2707,4.9898,COOLR location accuracy unavailable/uninterpre...


## 6. Save ML-eligible positive inventory


In [8]:
eligible_positive = positive.loc[
    positive["ml_eligible"]
].copy()

eligible_geojson = (
    CLEAN_DIR
    / "malaysia_positive_inventory_ml_eligible.geojson"
)

eligible_csv = (
    CLEAN_DIR
    / "malaysia_positive_inventory_ml_eligible.csv"
)

eligible_export = eligible_positive.copy()
eligible_export["event_date"] = (
    eligible_export["event_date"]
    .astype("string")
    .replace("<NA>", None)
)

eligible_export.to_file(
    eligible_geojson,
    driver="GeoJSON"
)

eligible_positive.drop(
    columns="geometry",
    errors="ignore"
).to_csv(
    eligible_csv,
    index=False
)

print("Saved:", eligible_geojson)
print("Saved:", eligible_csv)


Saved: C:\Landslide_FYP\Landslide_FYP_Development\02_cleaned_inventory\malaysia_positive_inventory_ml_eligible.geojson
Saved: C:\Landslide_FYP\Landslide_FYP_Development\02_cleaned_inventory\malaysia_positive_inventory_ml_eligible.csv


## 7. Build exclusion buffers around ML-eligible positives

For candidate background generation:

- minimum exclusion = 2 km
- COOLR uncertainty is respected up to 5 km
- JMG = 2 km

Because coarse >5 km COOLR records are not used as point-based training positives, they do not create huge training-area holes.


In [9]:
SAMPLING_CRS = "EPSG:6933"
MIN_BUFFER_M = 2_000

eligible_m = eligible_positive.to_crs(SAMPLING_CRS)
malaysia_m = malaysia.to_crs(SAMPLING_CRS)

if len(malaysia_m) == 1:
    malaysia_geom_m = malaysia_m.geometry.iloc[0]
else:
    malaysia_geom_m = malaysia_m.dissolve().geometry.iloc[0]

eligible_m["sampling_exclusion_m"] = MIN_BUFFER_M

coolr_eligible_mask = eligible_m["source_dataset"].eq("NASA_COOLR")

eligible_m.loc[
    coolr_eligible_mask,
    "sampling_exclusion_m"
] = np.maximum(
    MIN_BUFFER_M,
    eligible_m.loc[coolr_eligible_mask, "location_accuracy_m"]
)

buffers = [
    geom.buffer(float(distance))
    for geom, distance in zip(
        eligible_m.geometry,
        eligible_m["sampling_exclusion_m"]
    )
]

exclusion_union = unary_union(buffers)

sampling_area = malaysia_geom_m.difference(
    exclusion_union
)

malaysia_area_km2 = malaysia_geom_m.area / 1_000_000
sampling_area_km2 = sampling_area.area / 1_000_000
excluded_area_km2 = malaysia_area_km2 - sampling_area_km2

print(f"Malaysia area: {malaysia_area_km2:,.1f} km²")
print(f"Background eligible area: {sampling_area_km2:,.1f} km²")
print(f"Excluded buffer area: {excluded_area_km2:,.1f} km²")
print(f"Eligible share: {sampling_area_km2 / malaysia_area_km2 * 100:.2f}%")


Malaysia area: 329,819.9 km²
Background eligible area: 326,104.5 km²
Excluded buffer area: 3,715.4 km²
Eligible share: 98.87%


## 8. Create 50 km sampling grid


In [10]:
GRID_SIZE_M = 50_000

minx, miny, maxx, maxy = sampling_area.bounds

grid_geoms = []
grid_ids = []
counter = 1

x = math.floor(minx / GRID_SIZE_M) * GRID_SIZE_M

while x < maxx:
    y = math.floor(miny / GRID_SIZE_M) * GRID_SIZE_M

    while y < maxy:
        square = box(
            x,
            y,
            x + GRID_SIZE_M,
            y + GRID_SIZE_M
        )

        valid_part = square.intersection(sampling_area)

        if not valid_part.is_empty and valid_part.area > 0:
            grid_geoms.append(valid_part)
            grid_ids.append(f"GRID_{counter:04d}")
            counter += 1

        y += GRID_SIZE_M

    x += GRID_SIZE_M

grid = gpd.GeoDataFrame(
    {
        "grid_id": grid_ids,
        "geometry": grid_geoms,
    },
    geometry="geometry",
    crs=SAMPLING_CRS
)

grid["eligible_area_m2"] = grid.geometry.area

print("Eligible grid cells:", len(grid))


Eligible grid cells: 213


## 9. Generate a larger background candidate pool

Default:

**3,000 background candidates**

These are not yet final target=0 samples.


In [11]:
BACKGROUND_CANDIDATE_TARGET = 3000
MIN_CANDIDATE_SEPARATION_M = 1000
RANDOM_SEED = 42

rng = np.random.default_rng(RANDOM_SEED)


def proportional_allocation(weights, total_n):
    weights = np.asarray(weights, dtype=float)
    raw = weights / weights.sum() * total_n
    base = np.floor(raw).astype(int)

    remainder = int(total_n - base.sum())

    if remainder > 0:
        fraction = raw - base
        idx = np.argsort(-fraction)[:remainder]
        base[idx] += 1

    return base


grid["target_n"] = proportional_allocation(
    grid["eligible_area_m2"],
    BACKGROUND_CANDIDATE_TARGET
)

print("Allocated:", grid["target_n"].sum())


Allocated: 3000


In [12]:
def random_point(poly, rng):
    minx, miny, maxx, maxy = poly.bounds

    for _ in range(20000):
        p = Point(
            rng.uniform(minx, maxx),
            rng.uniform(miny, maxy)
        )

        if poly.covers(p):
            return p

    return None


def is_far_enough(point, existing, min_distance):
    for other in existing:
        if point.distance(other) < min_distance:
            return False
    return True


accepted = []
rows = []

for _, row in grid.iterrows():

    required = int(row["target_n"])
    made = 0
    attempts = 0

    cell_geom = row[grid.geometry.name]

    while made < required and attempts < 100000:
        attempts += 1

        p = random_point(cell_geom, rng)

        if p is None:
            continue

        if not is_far_enough(
            p,
            accepted,
            MIN_CANDIDATE_SEPARATION_M
        ):
            continue

        accepted.append(p)

        rows.append({
            "background_id": f"BG_{len(accepted):05d}",
            "grid_id": row["grid_id"],
            "sampling_seed": RANDOM_SEED,
            "geometry": p,
        })

        made += 1

    if made < required:
        print(
            f"WARNING {row['grid_id']}: requested {required}, generated {made}"
        )


background_m = gpd.GeoDataFrame(
    rows,
    geometry="geometry",
    crs=SAMPLING_CRS
)

print("Generated:", len(background_m))

if len(background_m) != BACKGROUND_CANDIDATE_TARGET:
    raise RuntimeError(
        "Background pool did not reach target."
    )


Generated: 3000


## 10. Convert candidate pool to WGS84 and save


In [13]:
background = background_m.to_crs("EPSG:4326")

background["longitude"] = background.geometry.x
background["latitude"] = background.geometry.y
background["sample_class"] = "background_candidate"
background["target"] = pd.NA

bg_geojson = (
    CLEAN_DIR
    / "background_candidates_3000.geojson"
)

bg_csv = (
    CLEAN_DIR
    / "background_candidates_3000.csv"
)

background.to_file(
    bg_geojson,
    driver="GeoJSON"
)

background.drop(
    columns="geometry",
    errors="ignore"
).to_csv(
    bg_csv,
    index=False
)

print("Saved:", bg_geojson)
print("Saved:", bg_csv)


Saved: C:\Landslide_FYP\Landslide_FYP_Development\02_cleaned_inventory\background_candidates_3000.geojson
Saved: C:\Landslide_FYP\Landslide_FYP_Development\02_cleaned_inventory\background_candidates_3000.csv


## 11. Spatial validation map


In [14]:
fig, ax = plt.subplots(figsize=(12, 9))

malaysia.boundary.plot(
    ax=ax,
    linewidth=0.8
)

background.plot(
    ax=ax,
    markersize=6,
    alpha=0.35,
    label="Background candidates"
)

eligible_positive.plot(
    ax=ax,
    markersize=30,
    marker="x",
    alpha=0.8,
    label="ML-eligible landslide"
)

ax.set_title(
    "ML-Eligible Positive Inventory and 3,000 Background Candidates"
)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.legend()
ax.grid(True, alpha=0.2)

plt.tight_layout()

map_path = (
    OUTPUT_DIR
    / "ml_eligible_positive_background_candidates_map.png"
)

fig.savefig(
    map_path,
    dpi=180,
    bbox_inches="tight"
)

plt.close(fig)

print("Saved:", map_path)


Saved: C:\Landslide_FYP\Landslide_FYP_Development\01_preprocessing\positive_accuracy_background_outputs\ml_eligible_positive_background_candidates_map.png


## 12. Stage-4B summary


In [15]:
result_summary = pd.DataFrame({
    "check": [
        "Historical reviewed positives",
        "ML-eligible positives",
        "Excluded coarse positives",
        "Maximum accepted COOLR accuracy m",
        "Background candidate target",
        "Background candidates generated",
        "Malaysia area km2",
        "Background eligible area km2",
        "Eligible share percent",
    ],
    "value": [
        len(positive),
        len(eligible_positive),
        len(excluded),
        MAX_COOLR_ACCURACY_M,
        BACKGROUND_CANDIDATE_TARGET,
        len(background),
        round(malaysia_area_km2, 2),
        round(sampling_area_km2, 2),
        round(sampling_area_km2 / malaysia_area_km2 * 100, 2),
    ]
})

display(result_summary)

result_summary.to_csv(
    OUTPUT_DIR / "stage_04B_summary.csv",
    index=False
)

print("\nNEXT STAGE")
print("----------")
print("05_background_candidate_feature_extraction.ipynb")
print()
print(
    "Extract terrain, land cover, road distance, "
    "waterway distance and other valid static predictors "
    "for ML-eligible positives + background candidates."
)


,check,value
0,Historical reviewed positives,287.00
1,ML-eligible positives,200.00
2,Excluded coarse positives,87.00
3,Maximum accepted COOLR accuracy m,5000.00
4,Background candidate target,3000.00
5,Background candidates generated,3000.00
6,Malaysia area km2,329819.88
7,Background eligible area km2,326104.46
8,Eligible share percent,98.87



NEXT STAGE
----------
05_background_candidate_feature_extraction.ipynb

Extract terrain, land cover, road distance, waterway distance and other valid static predictors for ML-eligible positives + background candidates.


# Output files

```text
Landslide_FYP_Development/
├── 01_preprocessing/
│   └── positive_accuracy_background_outputs/
│       ├── positive_excluded_from_point_ml.csv
│       ├── ml_eligible_positive_background_candidates_map.png
│       └── stage_04B_summary.csv
│
└── 02_cleaned_inventory/
    ├── malaysia_positive_inventory_ml_eligible.geojson
    ├── malaysia_positive_inventory_ml_eligible.csv
    ├── background_candidates_3000.geojson
    └── background_candidates_3000.csv
```

The original 287-event historical inventory remains unchanged.

This filtered inventory is only for fine-resolution point-based machine-learning feature extraction.
